# Exploration: Monza 2024 qualifying

Run `f1tel ingest --year 2024 --event Monza --session Q` first (needs a Formula 1 account, see the README).
Install the extras first: `pip install plotly jupyter`. This notebook reads the Parquet cache and uses the same alignment code as the API. Unofficial project; F1 and related marks belong to their owners.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

from f1tel.align import delta_time, resample_frame, ResampledLap

SESSION = Path('../data/raw/2024/Italian_Grand_Prix/Q')
laps = pd.read_parquet(SESSION / 'laps.parquet')
laps[laps.is_accurate].sort_values('lap_time_s').head(10)[['driver', 'lap_number', 'lap_time_s', 'compound']]

## Fastest lap per driver on the 10 m grid

In [ ]:
def fastest(driver):
    row = laps[(laps.driver == driver) & laps.is_accurate].nsmallest(1, 'lap_time_s').iloc[0]
    raw = pd.read_parquet(SESSION / driver / f'lap_{int(row.lap_number)}.parquet')
    g = resample_frame(raw, 10.0)
    return ResampledLap(g.distance_m.to_numpy(), g.time_s.to_numpy(), 10.0), g

drivers = laps.driver.unique()[:2]
(a, ga), (b, gb) = fastest(drivers[0]), fastest(drivers[1])
res = delta_time(a, b)
print(f'{drivers[1]} - {drivers[0]} at the line: {res.delta[-1]:+.3f} s (positive = {drivers[1]} slower)')

In [ ]:
fig = go.Figure(go.Scatter(x=res.distance, y=res.delta, name='delta'))
fig.update_layout(xaxis_title='Distance (m)', yaxis_title='Delta (s)')
fig